# Cat vs dog specialist: WideResNet-28-10 trained only on cat and dog images

A teacher for later distillation (Hinton et al. 2015, specialist models for confusable classes). It is never submitted: the final model stays the single 5M ResNeXt.

- Data: only the cat and dog images of our usual 45k training split (9,000 images), 2 classes. Validation and internal test: their cat and dog images (500 each).
- Recipe: same as our best runs (TrivialAugment, grayscale, crop + flip, cutout, ResizeMix, ASAM, EMA, SLA+SD with plain cross-entropy and beta = 1, cosine).
- Section 8 is the gate: it measures whether the specialist re-splitting cat vs dog improves our best model (`kd_student_resnext29_slasd_400ep_v2`, internal test 98.04). Only if it does, we use it as a teacher.

Needs an A100 (WRN-28-10 with 4 rotations is memory heavy). Run all; if Colab disconnects, Run all again and it resumes.

## 1. Config

In [1]:
CFG = dict(
    RUN_NAME="specialist_catdog_wrn28_10_v1",
    DRIVE_PROJECT_DIR="cifar10_resnet_challenge",
    SPLIT_FILE="split_seed42_val0.05_test0.05_n50000.npz",
    GENERALIST_RUN="kd_student_resnext29_slasd_400ep_v2",     # our best model (internal test 98.04), used only for the gate in section 8
    SEED=21,

    CLASSES=(3, 5),                    # CIFAR-10 ids: 3 = cat, 5 = dog

    # model: WideResNet-28-10 (Zagoruyko & Komodakis 2016), dropout 0.3 as in the WRN paper
    DEPTH=28, WIDEN=10, DROPOUT=0.3,

    # training
    EPOCHS=200, BATCH_SIZE=256, LR=0.1, MOMENTUM=0.9, WEIGHT_DECAY=5e-4, WARMUP_EPOCHS=5,
    ASAM_RHO=0.5, EMA_DECAY=0.999,
    BETA=1.0,                          # SLA+SD loss with the extra cross-entropy on the main head

    # augmentation
    TA_GROUP=16, GRAYSCALE_P=0.1, CUTOUT=8,
    MIX_PROB=0.5, RESIZEMIX_SCALE=(0.1, 0.8),
)
CFG

{'RUN_NAME': 'specialist_catdog_wrn28_10_v1',
 'DRIVE_PROJECT_DIR': 'cifar10_resnet_challenge',
 'SPLIT_FILE': 'split_seed42_val0.05_test0.05_n50000.npz',
 'GENERALIST_RUN': 'kd_student_resnext29_slasd_400ep_v2',
 'SEED': 21,
 'CLASSES': (3, 5),
 'DEPTH': 28,
 'WIDEN': 10,
 'DROPOUT': 0.3,
 'EPOCHS': 200,
 'BATCH_SIZE': 256,
 'LR': 0.1,
 'MOMENTUM': 0.9,
 'WEIGHT_DECAY': 0.0005,
 'WARMUP_EPOCHS': 5,
 'ASAM_RHO': 0.5,
 'EMA_DECAY': 0.999,
 'BETA': 1.0,
 'TA_GROUP': 16,
 'GRAYSCALE_P': 0.1,
 'CUTOUT': 8,
 'MIX_PROB': 0.5,
 'RESIZEMIX_SCALE': (0.1, 0.8)}

## 2. Setup

In [2]:
import os, time, math
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision.transforms import v2 as T
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
from google.colab import drive

drive.mount("/content/drive")
ROOT = os.path.join("/content/drive/MyDrive", CFG["DRIVE_PROJECT_DIR"])
CACHE = os.path.join(ROOT, "data", "cache")
RUN_DIR = os.path.join(ROOT, "runs", CFG["RUN_NAME"]); os.makedirs(RUN_DIR, exist_ok=True)
LAST, BEST = os.path.join(RUN_DIR, "last.pt"), os.path.join(RUN_DIR, "best.pt")

device = torch.device("cuda")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
torch.manual_seed(CFG["SEED"]); np.random.seed(CFG["SEED"])
print("GPU:", torch.cuda.get_device_name(0), "| run dir:", RUN_DIR)

def save(obj, path):
    torch.save(obj, path + ".tmp"); os.replace(path + ".tmp", path)
NC = len(CFG["CLASSES"])

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB | run dir: /content/drive/MyDrive/cifar10_resnet_challenge/runs/specialist_catdog_wrn28_10_v1


## 3. Data: cat and dog images only (same 45k / 2.5k / 2.5k split as every run)

In [3]:
X = np.load(os.path.join(CACHE, "train_x.npy")); y = np.load(os.path.join(CACHE, "train_y.npy"))
split = np.load(os.path.join(ROOT, "data", "splits", CFG["SPLIT_FILE"]))

def gpu(a): return torch.from_numpy(a).permute(0, 3, 1, 2).contiguous().to(device)

# full 10-class val / internal test (for the gate in section 8)
Xva_all, yva_all = gpu(X[split["val"]]), torch.from_numpy(y[split["val"]]).to(device)
Xte_all, yte_all = gpu(X[split["test"]]), torch.from_numpy(y[split["test"]]).to(device)

def catdog(idx):
    keep = idx[np.isin(y[idx], CFG["CLASSES"])]
    lab = np.searchsorted(CFG["CLASSES"], y[keep])            # cat -> 0, dog -> 1
    return gpu(X[keep]), torch.from_numpy(lab).to(device)
Xtr, ytr = catdog(split["train"]); Xva, yva = catdog(split["val"]); Xte, yte = catdog(split["test"])

# normalisation: same statistics as the full 45k training split (so the specialist sees images like our other models)
Xtr_full = gpu(X[split["train"]]).float() / 255
MEAN = Xtr_full.mean((0, 2, 3)).view(1, 3, 1, 1); STD = Xtr_full.std((0, 2, 3)).view(1, 3, 1, 1); del Xtr_full
def normalize(x): return ((x.float() / 255 - MEAN) / STD).contiguous(memory_format=torch.channels_last)
print("cat+dog images: train", len(Xtr), "| val", len(Xva), "| internal test", len(Xte), "| per class train", torch.bincount(ytr).tolist())

cat+dog images: train 9000 | val 500 | internal test 500 | per class train [4500, 4500]


## 4. Augmentation (same as our best runs)

In [4]:
trivial = T.TrivialAugmentWide()

def augment(x_u8):
    n, ar = len(x_u8), torch.arange(32, device=device)
    x_u8 = torch.cat([trivial(c) for c in x_u8.split(CFG["TA_GROUP"])])
    gray = (x_u8.float() * torch.tensor([0.299, 0.587, 0.114], device=device).view(1, 3, 1, 1)).sum(1, keepdim=True)
    g = torch.rand(n, device=device) < CFG["GRAYSCALE_P"]
    x_u8 = torch.where(g[:, None, None, None], gray.round().to(torch.uint8).expand(-1, 3, -1, -1), x_u8)
    x = (x_u8.float() / 255 - MEAN) / STD
    xp = F.pad(x, (4, 4, 4, 4), mode="reflect")
    dy, dx = torch.randint(0, 9, (n,), device=device), torch.randint(0, 9, (n,), device=device)
    x = xp[torch.arange(n, device=device)[:, None, None, None], torch.arange(3, device=device)[None, :, None, None],
           (dy[:, None] + ar)[:, None, :, None], (dx[:, None] + ar)[:, None, None, :]]
    flip = torch.rand(n, device=device) < 0.5
    x = torch.where(flip[:, None, None, None], x.flip(3), x)
    cy, cx, h = torch.randint(0, 32, (n,), device=device), torch.randint(0, 32, (n,), device=device), CFG["CUTOUT"] // 2
    hole = ((ar[None, :, None] - cy[:, None, None]).abs() < h) & ((ar[None, None, :] - cx[:, None, None]).abs() < h)
    return (x * (~hole)[:, None].float()).contiguous(memory_format=torch.channels_last)

def resizemix(x, y):
    """ResizeMix (Qin et al. 2020): paste a shrunk shuffled partner image; partner label weight = pasted area."""
    idx = torch.randperm(len(x), device=device)
    s = max(2, int(32 * np.random.uniform(*CFG["RESIZEMIX_SCALE"])))
    r, c = np.random.randint(0, 33 - s), np.random.randint(0, 33 - s)
    x = x.clone()
    x[:, :, r:r + s, c:c + s] = F.interpolate(x[idx], size=(s, s), mode="bilinear", align_corners=False)
    return x.contiguous(memory_format=torch.channels_last), y, y[idx], 1 - s * s / 1024

## 5. Model: WideResNet-28-10 + SLA head (2 classes x 4 rotations)

In [5]:
class WBlock(nn.Module):
    """Pre-activation wide residual block (Zagoruyko & Komodakis 2016)."""
    def __init__(self, cin, cout, stride, drop):
        super().__init__()
        self.bn1, self.conv1 = nn.BatchNorm2d(cin), nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn2, self.conv2 = nn.BatchNorm2d(cout), nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.drop = nn.Dropout(drop)
        self.skip = None if (stride == 1 and cin == cout) else nn.Conv2d(cin, cout, 1, stride, bias=False)
    def forward(self, x):
        o = F.relu(self.bn1(x))
        sc = x if self.skip is None else self.skip(o)
        o = self.conv2(self.drop(F.relu(self.bn2(self.conv1(o)))))
        return o + sc

class WRN(nn.Module):
    def __init__(self, depth, k, drop, nc):
        super().__init__()
        n = (depth - 4) // 6; w = [16, 16 * k, 32 * k, 64 * k]
        self.stem = nn.Conv2d(3, w[0], 3, 1, 1, bias=False)
        layers, cin = [], w[0]
        for i in range(3):
            for j in range(n):
                layers.append(WBlock(cin, w[i + 1], 2 if (j == 0 and i > 0) else 1, drop)); cin = w[i + 1]
        self.layers = nn.Sequential(*layers); self.bn = nn.BatchNorm2d(cin)
        self.fc = nn.Linear(cin, nc)             # main head (cat vs dog)
        self.fc_joint = nn.Linear(cin, nc * 4)   # SLA head: class x rotation, training only
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
    def features(self, x):
        return torch.flatten(F.adaptive_avg_pool2d(F.relu(self.bn(self.layers(self.stem(x)))), 1), 1)
    def forward(self, x): return self.fc(self.features(x))
    def both(self, x):
        f = self.features(x); return self.fc(f), self.fc_joint(f)

model = WRN(CFG["DEPTH"], CFG["WIDEN"], CFG["DROPOUT"], NC).to(device).to(memory_format=torch.channels_last)
print(f"WRN-{CFG['DEPTH']}-{CFG['WIDEN']} parameters: {sum(p.numel() for p in model.parameters()):,}  (teacher only, never submitted)")

WRN-28-10 parameters: 36,479,194  (teacher only, never submitted)


## 6. Optimizer, schedule, EMA, prediction

In [6]:
decay = [p for p in model.parameters() if p.ndim > 1]; no_decay = [p for p in model.parameters() if p.ndim <= 1]
opt = torch.optim.SGD([{"params": decay, "weight_decay": CFG["WEIGHT_DECAY"]}, {"params": no_decay, "weight_decay": 0.0}],
                      lr=CFG["LR"], momentum=CFG["MOMENTUM"], nesterov=True)
steps = math.ceil(len(Xtr) / CFG["BATCH_SIZE"]); warm = CFG["WARMUP_EPOCHS"] * steps
sched = torch.optim.lr_scheduler.SequentialLR(opt, [
    torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.01, total_iters=warm),
    torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["EPOCHS"] * steps - warm)], milestones=[warm])
ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(CFG["EMA_DECAY"]), use_buffers=True)

@torch.no_grad()
def predict(m, X, mode="sd", tta=True, nc=NC):
    """Probabilities. sd: main head | ag: SLA head averaged over 4 rotations | sdag: mean of both. tta: + flipped image."""
    m.eval(); out = []
    def probs(xb):
        if mode == "sd": return m(xb).float().softmax(1)
        ag = sum(m.both(torch.rot90(xb, k, (2, 3)))[1].float().view(-1, nc, 4)[:, :, k] for k in range(4)) / 4
        return ag.softmax(1) if mode == "ag" else 0.5 * (ag.softmax(1) + m(xb).float().softmax(1))
    for i in range(0, len(X), 500):
        xb = normalize(X[i:i + 500])
        with torch.autocast("cuda", dtype=torch.bfloat16):
            p = probs(xb)
            if tta: p = (p + probs(xb.flip(3))) / 2
        out.append(p)
    return torch.cat(out)

def acc(p, y): return (p.argmax(1) == y).float().mean().item()

## 7. Train (resumes from last.pt)

In [7]:
start, best_acc, history = 0, 0.0, []
if os.path.exists(LAST):
    ck = torch.load(LAST, map_location=device, weights_only=False)
    assert ck["cfg"] == CFG, "Config changed since this run started: use a new RUN_NAME"
    model.load_state_dict(ck["model"]); ema.load_state_dict(ck["ema"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
    start, best_acc, history = ck["epoch"], ck["best_acc"], ck["history"]
    print(f"resumed at epoch {start}, best val {best_acc:.4f}")

params = list(model.parameters()); rot = torch.arange(4, device=device)

def loss_fn(x4, ya, yb, lam, B):
    """SLA+SD (Lee et al. 2020, eq. 5), plain cross-entropy, beta term on the main head."""
    with torch.autocast("cuda", dtype=torch.bfloat16):
        sd_lo, j_lo = model.both(x4)
    sd_lo, j_lo = sd_lo.float(), j_lo.float()
    joint = lambda t: F.cross_entropy(j_lo, t.repeat(4) * 4 + rot.repeat_interleave(B))
    l_sla = lam * joint(ya) + (1 - lam) * joint(yb)
    agg = torch.stack([j_lo.view(4, B, NC, 4)[k, :, :, k] for k in range(4)]).mean(0)
    sd0 = sd_lo[:B]
    l_kl = F.kl_div(F.log_softmax(sd0, 1), F.softmax(agg.detach(), 1), reduction="batchmean")
    l_ce = lam * F.cross_entropy(sd0, ya) + (1 - lam) * F.cross_entropy(sd0, yb)
    return sd0, l_sla + l_kl + CFG["BETA"] * l_ce

for epoch in range(start, CFG["EPOCHS"]):
    t0 = time.time(); model.train()
    tot, correct = torch.zeros((), device=device), torch.zeros((), device=device)
    perm = torch.randperm(len(Xtr), device=device)
    for i in range(0, len(Xtr), CFG["BATCH_SIZE"]):
        idx = perm[i:i + CFG["BATCH_SIZE"]]; B = len(idx)
        x, ya, yb, lam = augment(Xtr[idx]), ytr[idx], ytr[idx], 1.0
        if np.random.rand() < CFG["MIX_PROB"]:
            x, ya, yb, lam = resizemix(x, ya)
        x4 = torch.cat([torch.rot90(x, k, (2, 3)) for k in range(4)]).contiguous(memory_format=torch.channels_last)
        opt.zero_grad(set_to_none=True)
        out, loss = loss_fn(x4, ya, yb, lam, B); loss.backward()
        with torch.no_grad():                                   # ASAM (Kwon et al. 2021)
            tw = [p.abs() + 0.01 for p in params]
            norm = torch.norm(torch.stack([(t * p.grad).norm() for t, p in zip(tw, params) if p.grad is not None])) + 1e-12
            eps = [t * t * p.grad * (CFG["ASAM_RHO"] / norm) if p.grad is not None else None for t, p in zip(tw, params)]
            for p, e in zip(params, eps):
                if e is not None: p.add_(e)
        opt.zero_grad(set_to_none=True)
        loss_fn(x4, ya, yb, lam, B)[1].backward()
        with torch.no_grad():
            for p, e in zip(params, eps):
                if e is not None: p.sub_(e)
        opt.step(); sched.step(); ema.update_parameters(model)
        tot += loss.detach() * B; correct += (out.argmax(1) == ya).sum()

    val_acc = acc(predict(ema.module, Xva, tta=False), yva)
    history.append({"epoch": epoch + 1, "loss": tot.item() / len(Xtr), "train_acc": correct.item() / len(Xtr), "val_acc": val_acc,
                    "lr": sched.get_last_lr()[0], "sec": time.time() - t0})
    if val_acc > best_acc:
        best_acc = val_acc
        save({"model": ema.module.state_dict(), "cfg": CFG, "epoch": epoch + 1, "val_acc": val_acc,
              "norm": {"mean": MEAN.cpu(), "std": STD.cpu()}}, BEST)
    save({"model": model.state_dict(), "ema": ema.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
          "epoch": epoch + 1, "best_acc": best_acc, "history": history, "cfg": CFG}, LAST)
    h = history[-1]
    print(f"ep {epoch+1:3d}/{CFG['EPOCHS']} | loss {h['loss']:.4f} | train acc {h['train_acc']:.4f} | val cat-vs-dog (EMA) {val_acc:.4f}"
          f"{' *' if best_acc == val_acc else '  '} | lr {h['lr']:.4f} | {h['sec']:.0f}s | eta {(CFG['EPOCHS']-epoch-1)*h['sec']/3600:.1f} h")
pd.DataFrame(history).to_csv(os.path.join(RUN_DIR, "history.csv"), index=False)
print("done, best val cat-vs-dog", best_acc)

ep   1/200 | loss 2.9594 | train acc 0.4978 | val cat-vs-dog (EMA) 0.5020 * | lr 0.0208 | 34s | eta 1.9 h
ep   2/200 | loss 2.9215 | train acc 0.4933 | val cat-vs-dog (EMA) 0.5000   | lr 0.0406 | 17s | eta 0.9 h
ep   3/200 | loss 2.8026 | train acc 0.5004 | val cat-vs-dog (EMA) 0.5000   | lr 0.0604 | 17s | eta 0.9 h
ep   4/200 | loss 2.7984 | train acc 0.5050 | val cat-vs-dog (EMA) 0.5000   | lr 0.0802 | 17s | eta 0.9 h
ep   5/200 | loss 2.7820 | train acc 0.4951 | val cat-vs-dog (EMA) 0.5000   | lr 0.1000 | 17s | eta 0.9 h
ep   6/200 | loss 2.7696 | train acc 0.4966 | val cat-vs-dog (EMA) 0.5000   | lr 0.1000 | 17s | eta 0.9 h
ep   7/200 | loss 2.7590 | train acc 0.5139 | val cat-vs-dog (EMA) 0.5000   | lr 0.1000 | 17s | eta 0.9 h
ep   8/200 | loss 2.7479 | train acc 0.5161 | val cat-vs-dog (EMA) 0.5000   | lr 0.0999 | 17s | eta 0.9 h
ep   9/200 | loss 2.7238 | train acc 0.5333 | val cat-vs-dog (EMA) 0.5000   | lr 0.0999 | 17s | eta 0.9 h
ep  10/200 | loss 2.7155 | train acc 0.5523 | 

## 8. Gate: does the specialist improve our best model?

Loads our best model (98.04 internal test) and compares, on the same images:
1. cat-vs-dog accuracy on the true cat and dog images: specialist vs best model
2. full 10-class accuracy of the best model, before and after re-splitting its cat+dog probability with the specialist (the way the specialist would act as a teacher)

In [8]:
spec = WRN(CFG["DEPTH"], CFG["WIDEN"], CFG["DROPOUT"], NC).to(device).to(memory_format=torch.channels_last)
sb = torch.load(BEST, map_location=device, weights_only=False); spec.load_state_dict(sb["model"]); spec.eval()
s_val = {m: acc(predict(spec, Xva, m), yva) for m in ("sd", "ag", "sdag")}
S_MODE = max(s_val, key=s_val.get)
print(f"specialist best.pt epoch {sb['epoch']} | val cat-vs-dog: {s_val} -> using {S_MODE}")

# --- our best model (architecture as saved by the research notebook) ---
class ResNeXtBlock(nn.Module):
    def __init__(self, cin, inner, cout, stride, cardinality):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, inner, 1, bias=False); self.bn1 = nn.BatchNorm2d(inner)
        self.conv2 = nn.Conv2d(inner, inner, 3, stride, 1, groups=cardinality, bias=False); self.bn2 = nn.BatchNorm2d(inner)
        self.conv3 = nn.Conv2d(inner, cout, 1, bias=False); self.bn3 = nn.BatchNorm2d(cout)
        self.shortcut = nn.Identity()
        if stride != 1 or cin != cout:
            self.shortcut = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = F.relu(self.bn2(self.conv2(out)), inplace=True)
        return F.relu(self.bn3(self.conv3(out)) + self.shortcut(x), inplace=True)

class ResNeXt29(nn.Module):
    def __init__(self, cardinality, bottleneck_width, blocks, out_widths):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 64, 3, 1, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        layers, cin = [], 64
        for i, (cout, nb) in enumerate(zip(out_widths, blocks)):
            for j in range(nb):
                layers.append(ResNeXtBlock(cin, cardinality * bottleneck_width * 2 ** i, cout, 2 if (j == 0 and i > 0) else 1, cardinality)); cin = cout
        self.layers = nn.Sequential(*layers); self.drop = nn.Dropout(0.0)
        self.fc = nn.Linear(cin, 10); self.fc_joint = nn.Linear(cin, 40)
    def features(self, x): return torch.flatten(F.adaptive_avg_pool2d(self.layers(self.stem(x)), 1), 1)
    def forward(self, x): return self.fc(self.features(x))
    def both(self, x):
        f = self.features(x); return self.fc(f), self.fc_joint(f)

gb = torch.load(os.path.join(ROOT, "runs", CFG["GENERALIST_RUN"], "best.pt"), map_location=device, weights_only=False)
gc = gb["cfg"]
gen = ResNeXt29(gc["CARDINALITY"], gc["BOTTLENECK_WIDTH"], gc["BLOCKS"], gc["WIDTHS"]).to(device).to(memory_format=torch.channels_last)
gen.load_state_dict(gb["model"]); gen.eval()
assert torch.allclose(gb["norm"]["mean"].to(device), MEAN, atol=1e-4), "normalisation differs from the generalist"
g_val = {m: acc(predict(gen, Xva_all, m, nc=10), yva_all) for m in ("sd", "ag", "sdag")}
G_MODE = max(g_val, key=g_val.get)
print(f"best model val (10-class): {g_val} -> using {G_MODE}")

c, d = CFG["CLASSES"]
def report(name, Xall, yall, Xcd, ycd):
    pg = predict(gen, Xall, G_MODE, nc=10)
    m = (yall == c) | (yall == d)
    g_cd = ((pg[m][:, d] > pg[m][:, c]).long() == ycd).float().mean().item()       # best model: cat vs dog only
    s_cd = acc(predict(spec, Xcd, S_MODE), ycd)
    ps = predict(spec, Xall, S_MODE)                                               # specialist on every image
    mass = pg[:, c] + pg[:, d]
    pc = pg.clone(); pc[:, c], pc[:, d] = mass * ps[:, 0], mass * ps[:, 1]          # re-split cat+dog share
    print(f"\n{name}: cat-vs-dog on {int(m.sum())} cat/dog images | best model {g_cd:.4f} ({int(round((1-g_cd)*m.sum().item()))} wrong)"
          f" | specialist {s_cd:.4f} ({int(round((1-s_cd)*m.sum().item()))} wrong)")
    print(f"{name}: 10-class accuracy | best model {acc(pg, yall):.4f} ({int((pg.argmax(1) != yall).sum())} errors)"
          f" | best model + specialist re-split {acc(pc, yall):.4f} ({int((pc.argmax(1) != yall).sum())} errors)")
report("VALIDATION", Xva_all, yva_all, Xva, yva)
report("INTERNAL TEST", Xte_all, yte_all, Xte, yte)
print("\nUse the specialist as a teacher only if the re-split improves both validation and internal test.")

specialist best.pt epoch 196 | val cat-vs-dog: {'sd': 0.9460000395774841, 'ag': 0.9440000653266907, 'sdag': 0.9480000734329224} -> using sdag
best model val (10-class): {'sd': 0.9791999459266663, 'ag': 0.9811999797821045, 'sdag': 0.9795999526977539} -> using ag

VALIDATION: cat-vs-dog on 500 cat/dog images | best model 0.9760 (12 wrong) | specialist 0.9480 (26 wrong)
VALIDATION: 10-class accuracy | best model 0.9812 (47 errors) | best model + specialist re-split 0.9756 (61 errors)

INTERNAL TEST: cat-vs-dog on 500 cat/dog images | best model 0.9740 (13 wrong) | specialist 0.9360 (32 wrong)
INTERNAL TEST: 10-class accuracy | best model 0.9804 (49 errors) | best model + specialist re-split 0.9728 (68 errors)

Use the specialist as a teacher only if the re-split improves both validation and internal test.
